# Final Project - SageMaker Monitoring Notebook (Fixed)

Hiring-bias / Adult Income project.

Covers:
- Endpoint deployment with data capture
- Data quality monitoring
- Model quality monitoring
- SageMaker Clarify (bias drift + explainability)
- Traffic generation against the live endpoint
- Infrastructure monitoring with CloudWatch
- CloudWatch dashboard creation

Run in SageMaker Studio. All bucket names and S3 paths now match Notebooks 01 and 02.

## 1. Imports and config

In [ ]:
import os
import json
import time
import boto3
import sagemaker
import pandas as pd
import numpy as np

from datetime import datetime
from sagemaker import get_execution_role
from sagemaker.model_monitor import (
    DataCaptureConfig,
    DefaultModelMonitor,
    DatasetFormat,
    ModelQualityMonitor,
    EndpointInput,
    CronExpressionGenerator,
)
from sagemaker.model_monitor import ModelBiasMonitor, ModelExplainabilityMonitor
from sagemaker.clarify import (
    BiasConfig,
    DataConfig,
    ModelConfig,
    ModelPredictedLabelConfig,
    SHAPConfig,
)

region = boto3.Session().region_name
session = sagemaker.Session()
s3_client = boto3.client("s3", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)
cw_client = boto3.client("cloudwatch", region_name=region)
sts_client = boto3.client("sts", region_name=region)

account_id = sts_client.get_caller_identity()["Account"]

try:
    role_arn = get_execution_role()
except Exception:
    role_arn = "REPLACE_WITH_YOUR_SAGEMAKER_EXECUTION_ROLE_ARN"

# Match Notebooks 01 and 02 exactly
bucket_name = "hiring-bias-adult-income-mustafa"
data_prefix = "hiring-bias-project"
project_prefix = "hiring-bias-final"

endpoint_name = f"{project_prefix}-endpoint"
model_name = f"{project_prefix}-model"
endpoint_config_name = f"{project_prefix}-endpoint-config"

monitoring_output_prefix = f"{project_prefix}/monitoring"
baseline_prefix = f"{project_prefix}/baseline"
reports_prefix = f"{project_prefix}/reports"
ground_truth_prefix = f"{project_prefix}/ground-truth"
clarify_prefix = f"{project_prefix}/clarify"

# Reuse the consolidated train.csv that Notebook 02 wrote
baseline_data_s3_uri = f"s3://{bucket_name}/{data_prefix}/data/pipeline-input/train.csv"

# The trained model artifact uploaded by Notebook 02's local-training cell
model_artifact_s3 = f"s3://{bucket_name}/{data_prefix}/models/local-train/model.tar.gz"

# The processed test set (with protected attribute columns stripped before scoring)
local_test_s3_key = f"{data_prefix}/data/local-processed/test.csv"
test_data_s3 = f"s3://{bucket_name}/{local_test_s3_key}"

ground_truth_s3_uri = f"s3://{bucket_name}/{ground_truth_prefix}/"

print("Region:", region)
print("Account ID:", account_id)
print("Bucket:", bucket_name)
print("Role ARN:", role_arn)
print("Endpoint Name:", endpoint_name)
print("Baseline data:", baseline_data_s3_uri)
print("Model artifact:", model_artifact_s3)

## 2. Deploy the endpoint with data capture enabled

In [ ]:
# DataCaptureConfig writes every request and response to S3 so the monitor jobs
# have something to compare against the baseline
data_capture_prefix = f"{project_prefix}/datacapture"
data_capture_s3_uri = f"s3://{bucket_name}/{data_capture_prefix}"

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=data_capture_s3_uri,
    capture_options=["REQUEST", "RESPONSE"],
)

print("Data capture S3 URI:", data_capture_s3_uri)

In [ ]:
# Deploy from the model artifact that Notebook 02 uploaded.
# Using sagemaker.model.Model directly because the artifact was trained locally
# (training job quota was 0), so we don't have an Estimator to .deploy() from
from sagemaker.model import Model
from sagemaker.image_uris import retrieve

image_uri = retrieve(
    framework="xgboost",
    region=region,
    version="1.7-1",
    py_version="py3",
    instance_type="ml.m5.large",
)

model = Model(
    image_uri=image_uri,
    model_data=model_artifact_s3,
    role=role_arn,
    sagemaker_session=session,
    name=model_name,
)

# Check first - skip deploy if endpoint is already InService
try:
    endpoint_desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
    print("Endpoint already exists:", endpoint_desc["EndpointStatus"])
except sm_client.exceptions.ClientError:
    print("Deploying new endpoint - takes 5-8 minutes...")
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type="ml.m5.large",
        endpoint_name=endpoint_name,
        data_capture_config=data_capture_config,
    )
    print("Endpoint deployed:", endpoint_name)

## 3. Generate traffic against the endpoint

Monitor schedules need captured invocations to compare against the baseline.
Without sending requests, the dashboard will be empty and the schedule will
report no executions. This cell sends a few hundred real records from the
processed test set so the captures exist.

In [ ]:
runtime = boto3.client("sagemaker-runtime", region_name=region)

# Pull the processed test CSV down, strip off the label and protected columns
# (the model only sees raw features at inference time)
s3_client.download_file(bucket_name, local_test_s3_key, "test_for_traffic.csv")
test_for_traffic = pd.read_csv("test_for_traffic.csv", header=None)

n_protected = 3
# Column 0 is the label, columns 1 to -n_protected are features
features_only = test_for_traffic.iloc[:, 1:-n_protected]

# Send 300 invocations to populate the data capture S3 prefix
N_INVOCATIONS = 300
sample = features_only.sample(n=N_INVOCATIONS, random_state=42, replace=True)

for i, row in enumerate(sample.itertuples(index=False)):
    payload = ",".join(str(v) for v in row)
    runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="text/csv",
        Body=payload,
    )
    if (i + 1) % 50 == 0:
        print(f"  sent {i+1}/{N_INVOCATIONS} invocations")
    time.sleep(0.05)

print(f"Done. {N_INVOCATIONS} requests sent. Captures will appear under {data_capture_s3_uri}")
print("Wait 5-10 minutes for captures to land in S3 before running the baseline jobs.")

## 4. Data quality monitoring - baseline

In [ ]:
baseline_results_uri = f"s3://{bucket_name}/{baseline_prefix}/data-quality"
monitor_output_uri = f"s3://{bucket_name}/{monitoring_output_prefix}/data-quality"

my_default_monitor = DefaultModelMonitor(
    role=role_arn,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session,
)

my_default_monitor.suggest_baseline(
    baseline_dataset=baseline_data_s3_uri,
    dataset_format=DatasetFormat.csv(header=False),
    output_s3_uri=baseline_results_uri,
    wait=True,
)

latest_baselining_job = my_default_monitor.latest_baselining_job
print("Baselining job name:", latest_baselining_job.job_name)
print("Statistics file:", latest_baselining_job.baseline_statistics().s3_uri)
print("Constraints file:", latest_baselining_job.suggested_constraints().s3_uri)

## 5. Data quality monitoring - schedule

In [ ]:
data_quality_schedule_name = f"{project_prefix}-data-quality-schedule"

my_default_monitor.create_monitoring_schedule(
    monitor_schedule_name=data_quality_schedule_name,
    endpoint_input=EndpointInput(endpoint_name=endpoint_name),
    output_s3_uri=monitor_output_uri,
    statistics=my_default_monitor.baseline_statistics(),
    constraints=my_default_monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print("Data quality schedule created:", data_quality_schedule_name)

## 6. Model quality monitoring (predictions vs ground truth)

In [ ]:
model_quality_output_uri = f"s3://{bucket_name}/{monitoring_output_prefix}/model-quality"
model_quality_schedule_name = f"{project_prefix}-model-quality-schedule"
model_quality_baseline_uri = f"s3://{bucket_name}/{baseline_prefix}/model-quality"

model_quality_monitor = ModelQualityMonitor(
    role=role_arn,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
    sagemaker_session=session,
)

# Build a baseline file with predictions + ground truth in the format the monitor expects
# Pull predictions from the local model
import xgboost as xgb
import tarfile

s3_client.download_file(bucket_name, f"{data_prefix}/models/local-train/model.tar.gz", "model.tar.gz")
with tarfile.open("model.tar.gz") as tar:
    tar.extractall(path="model_for_baseline")

bst = xgb.Booster()
bst.load_model("model_for_baseline/xgboost-model")

s3_client.download_file(bucket_name, local_test_s3_key, "test_for_baseline.csv")
test_df = pd.read_csv("test_for_baseline.csv", header=None)
y_true = test_df.iloc[:, 0].values
X_features = test_df.iloc[:, 1:-n_protected].values

dmat = xgb.DMatrix(X_features)
y_proba = bst.predict(dmat)
y_pred = (y_proba >= 0.5).astype(int)

baseline_df = pd.DataFrame({
    "prediction": y_pred,
    "probability": y_proba,
    "label": y_true,
})
baseline_df.to_csv("model_quality_baseline.csv", index=False)
s3_client.upload_file("model_quality_baseline.csv", bucket_name,
                      f"{baseline_prefix}/model-quality/baseline.csv")
mq_baseline_s3 = f"s3://{bucket_name}/{baseline_prefix}/model-quality/baseline.csv"
print("Model quality baseline file:", mq_baseline_s3)

In [ ]:
model_quality_monitor.suggest_baseline(
    baseline_dataset=mq_baseline_s3,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=model_quality_baseline_uri,
    problem_type="BinaryClassification",
    inference_attribute="prediction",
    probability_attribute="probability",
    ground_truth_attribute="label",
    wait=True,
)

print("Model quality baseline job complete.")

### Ground truth file for the schedule

In production, ground truth would arrive later (the actual hiring outcome).
For the demo we upload a synthetic ground truth file that matches the invocations
we sent in step 3, so the schedule has something to merge against.

In [ ]:
# Build a ground truth file the schedule can consume.
# Format: {eventId, groundTruthData} JSON-lines, one per captured event.
# In production, replace this with real outcome data.

gt_records = []
for i in range(N_INVOCATIONS):
    gt_records.append({
        "groundTruthData": {
            "data": int(y_true[i % len(y_true)]),
            "encoding": "CSV",
        },
        "eventMetadata": {
            "eventId": f"event-{i}",
        },
        "eventVersion": "0",
    })

# Ground truth must be organized by hour: <prefix>/YYYY/MM/DD/HH/<filename>.jsonl
from datetime import datetime, timezone
now = datetime.now(timezone.utc)
hour_path = now.strftime("%Y/%m/%d/%H")
gt_filename = "ground_truth.jsonl"

with open(gt_filename, "w") as f:
    for r in gt_records:
        f.write(json.dumps(r) + "\n")

gt_key = f"{ground_truth_prefix}/{hour_path}/{gt_filename}"
s3_client.upload_file(gt_filename, bucket_name, gt_key)
print("Ground truth uploaded to:", f"s3://{bucket_name}/{gt_key}")

In [ ]:
model_quality_monitor.create_monitoring_schedule(
    monitor_schedule_name=model_quality_schedule_name,
    endpoint_input=EndpointInput(
        endpoint_name=endpoint_name,
        probability_attribute="0",
        inference_attribute="0",
    ),
    output_s3_uri=model_quality_output_uri,
    problem_type="BinaryClassification",
    ground_truth_input=ground_truth_s3_uri,
    constraints=model_quality_monitor.suggested_constraints(),
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print("Model quality schedule created:", model_quality_schedule_name)

## 7. SageMaker Clarify - bias drift and explainability

This is the bias audit piece. Clarify computes pre-training and post-training
bias metrics (Disparate Impact, DPL, SPD, etc.) and SHAP explainability values
against the deployed endpoint.

In [ ]:
# Build a Clarify-format dataset: features + label + protected attribute names

# Pull the consolidated train.csv to know the column names Clarify will reference
train_for_clarify = f"s3://{bucket_name}/{data_prefix}/data/train/train.csv"
s3_client.download_file(bucket_name, f"{data_prefix}/data/train/train.csv", "train_for_clarify.csv")
clarify_df = pd.read_csv("train_for_clarify.csv")

# Clarify expects raw features in a CSV
clarify_cols = ["age", "workclass", "fnlwgt", "education", "education_num",
                "marital_status", "occupation", "relationship", "race", "sex",
                "capital_gain", "capital_loss", "hours_per_week", "native_country"]
label_col = "income_binary"

clarify_input = clarify_df[[label_col] + clarify_cols].copy()
clarify_input.to_csv("clarify_input.csv", index=False)

clarify_input_s3 = f"{clarify_prefix}/clarify_input.csv"
s3_client.upload_file("clarify_input.csv", bucket_name, clarify_input_s3)
clarify_data_uri = f"s3://{bucket_name}/{clarify_input_s3}"
clarify_output_uri = f"s3://{bucket_name}/{clarify_prefix}/output"
print("Clarify input:", clarify_data_uri)

In [ ]:
from sagemaker import clarify

clarify_processor = clarify.SageMakerClarifyProcessor(
    role=role_arn,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    sagemaker_session=session,
)

data_config = clarify.DataConfig(
    s3_data_input_path=clarify_data_uri,
    s3_output_path=clarify_output_uri,
    label=label_col,
    headers=[label_col] + clarify_cols,
    dataset_type="text/csv",
)

# Audit bias against sex with "Male" as the privileged group
bias_config_sex = clarify.BiasConfig(
    label_values_or_threshold=[1],          # >50K is the favorable outcome
    facet_name="sex",
    facet_values_or_threshold=["Female"],   # unprivileged
    group_name="race",                       # also break down by race
)

model_config = clarify.ModelConfig(
    model_name=model_name,
    instance_type="ml.m5.large",
    instance_count=1,
    accept_type="text/csv",
    content_type="text/csv",
)

predictions_config = clarify.ModelPredictedLabelConfig(probability_threshold=0.5)

# Pre-training + post-training bias against sex
clarify_processor.run_bias(
    data_config=data_config,
    bias_config=bias_config_sex,
    model_config=model_config,
    model_predicted_label_config=predictions_config,
    pre_training_methods="all",
    post_training_methods="all",
)

print("Clarify bias report written to:", clarify_output_uri)

In [ ]:
# SHAP explainability across the test set
shap_config = clarify.SHAPConfig(
    baseline=[clarify_input[clarify_cols].mode().iloc[0].tolist()],
    num_samples=100,
    agg_method="mean_abs",
    save_local_shap_values=True,
)

explainability_output_uri = f"s3://{bucket_name}/{clarify_prefix}/explainability"

explainability_data_config = clarify.DataConfig(
    s3_data_input_path=clarify_data_uri,
    s3_output_path=explainability_output_uri,
    label=label_col,
    headers=[label_col] + clarify_cols,
    dataset_type="text/csv",
)

clarify_processor.run_explainability(
    data_config=explainability_data_config,
    model_config=model_config,
    explainability_config=shap_config,
)

print("Explainability report written to:", explainability_output_uri)

## 8. CloudWatch dashboard

In [ ]:
dashboard_name = f"{project_prefix}-cloudwatch-dashboard"

widgets = [
    {
        "type": "metric",
        "x": 0, "y": 0, "width": 12, "height": 6,
        "properties": {
            "metrics": [
                ["AWS/SageMaker", "Invocations", "EndpointName", endpoint_name],
                [".", "Invocation4XXErrors", ".", "."],
                [".", "Invocation5XXErrors", ".", "."],
            ],
            "view": "timeSeries",
            "stacked": False,
            "region": region,
            "title": "Endpoint Requests and Errors",
            "period": 300,
            "stat": "Sum",
        },
    },
    {
        "type": "metric",
        "x": 12, "y": 0, "width": 12, "height": 6,
        "properties": {
            "metrics": [
                ["AWS/SageMaker", "ModelLatency", "EndpointName", endpoint_name],
                [".", "OverheadLatency", ".", "."],
            ],
            "view": "timeSeries",
            "stacked": False,
            "region": region,
            "title": "Endpoint Latency",
            "period": 300,
            "stat": "Average",
        },
    },
    {
        "type": "metric",
        "x": 0, "y": 6, "width": 24, "height": 6,
        "properties": {
            "metrics": [
                ["AWS/SageMaker/ModelMonitoring", "violations",
                 "MonitoringScheduleName", data_quality_schedule_name],
                [".", "Executions", ".", "."],
                [".", "ProcessedSamples", ".", "."],
                ["AWS/SageMaker/ModelMonitoring", "violations",
                 "MonitoringScheduleName", model_quality_schedule_name],
            ],
            "view": "timeSeries",
            "stacked": False,
            "region": region,
            "title": "Model Monitor Violations and Executions",
            "period": 300,
            "stat": "Sum",
        },
    },
]

cw_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps({"widgets": widgets}),
)

print("CloudWatch dashboard created:", dashboard_name)

## 9. Verification summary

In [ ]:
for schedule_name in [data_quality_schedule_name, model_quality_schedule_name]:
    try:
        desc = sm_client.describe_monitoring_schedule(MonitoringScheduleName=schedule_name)
        print("\nSchedule:", schedule_name)
        print("Status:", desc.get("MonitoringScheduleStatus"))
        last_exec = desc.get("LastMonitoringExecutionSummary", {})
        if last_exec:
            print("Last execution status:", last_exec.get("MonitoringExecutionStatus"))
    except Exception as e:
        print(f"Could not describe {schedule_name}: {e}")

summary = {
    "project": "Hiring Bias / Adult Income Monitoring",
    "bucket": bucket_name,
    "endpoint_name": endpoint_name,
    "model_artifact": model_artifact_s3,
    "data_capture_s3_uri": data_capture_s3_uri,
    "baseline_data": baseline_data_s3_uri,
    "data_quality_schedule": data_quality_schedule_name,
    "model_quality_schedule": model_quality_schedule_name,
    "clarify_output": f"s3://{bucket_name}/{clarify_prefix}",
    "cloudwatch_dashboard": dashboard_name,
    "ground_truth_prefix": ground_truth_s3_uri,
    "generated_at": datetime.utcnow().isoformat() + "Z",
}

print("\nSummary:")
print(json.dumps(summary, indent=2))

## 10. Cleanup (run this when done with the demo)

These resources cost money on an hourly basis - delete after the demo recording.

In [ ]:
# Uncomment to clean up

# sm_client.delete_monitoring_schedule(MonitoringScheduleName=data_quality_schedule_name)
# sm_client.delete_monitoring_schedule(MonitoringScheduleName=model_quality_schedule_name)
# sm_client.delete_endpoint(EndpointName=endpoint_name)
# sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
# sm_client.delete_model(ModelName=model_name)
# cw_client.delete_dashboards(DashboardNames=[dashboard_name])
# print("Cleanup done")